### Note that this notebook was run in Anaconda and not Colab because colab cannot access files on your system until they are uploaded to the cloud. Due to dataset size that could not be uploaded to google drive before processing, Anaconda was used instead. the notebook was added here simply for accessibility.

# **Impiorting the required Library**

In [ ]:
import zipfile
import pandas as pd
import os
import shutil
import io

# **Data Loading**

In [ ]:
mri_folder = "F:/User/Downloads/ADNI1.zip"
output_folder = "F:/User/Downloads/ADNI1_Dataset"
csv_file = "F:/User/Downloads/ADNI1csv.csv"

In [ ]:
for diag in ['CN', 'MCI', 'AD']:
    os.makedirs(os.path.join(output_folder, diag), exist_ok=True)

In [ ]:
df = pd.read_csv(csv_file)

# Inspect columns
print("CSV columns:", df.columns.tolist())

# Keep baseline visit only
baseline_df = df[df['Visit'] == 'sc']

# Create dictionary: Subject → Group (diagnosis)
subject_to_group = dict(zip(baseline_df['Subject'], baseline_df['Group']))

print(f"Number of baseline subjects: {len(subject_to_group)}")

CSV columns: ['Image Data ID', 'Subject', 'Group', 'Sex', 'Age', 'Visit', 'Modality', 'Description', 'Type', 'Acq Date', 'Format', 'Downloaded']
Number of baseline subjects: 639


In [ ]:
if mri_folder.endswith(".zip"):
    extracted_folder_name = os.path.splitext(os.path.basename(mri_folder))[0]
    extracted_path = os.path.join(os.path.dirname(mri_folder) if os.path.dirname(mri_folder) else '.', extracted_folder_name)

    if not os.path.exists(extracted_path):
        print(f"Extracting {mri_folder} to {extracted_path}...")
        with zipfile.ZipFile(mri_folder, 'r') as zip_ref:
            zip_ref.extractall(extracted_path)
        print("Extraction complete.")
    else:
        print(f"Directory {extracted_path} already exists. Skipping extraction.")
    mri_folder_to_process = extracted_path
else:
    mri_folder_to_process = mri_folder

Extracting F:/User/Downloads/ADNI1.zip to F:/User/Downloads\ADNI1...
Extraction complete.


In [ ]:
count = 0
for root, dirs, files in os.walk(mri_folder_to_process):
    for file in files:
        if file.endswith(".nii") or file.endswith(".nii.gz"):
            matched = False
            for subject, group in subject_to_group.items():
                # Check if the Subject string is in the filename
                if subject in file:
                    src = os.path.join(root, file)
                    dst = os.path.join(output_folder, group, file)
                    shutil.copy(src, dst)  # Use shutil.move() if you want to move instead of copy
                    count += 1
                    matched = True
                    break
            if not matched:
                print(f"Warning: No match in CSV for {file}")

print(f"✅ Done! {count} MRI files sorted into CN/MCI/AD folders in {output_folder}")

✅ Done! 2294 MRI files sorted into CN/MCI/AD folders in F:/User/Downloads/ADNI1_Dataset
